# Tuning `tau_edge` for Neighbour-Vote Assignment

**Goal:** Find the best cosine-similarity threshold (`tau_edge`) to maximise the
precision/recall tradeoff between *coverage* (how many posts get assigned a cluster)
and *coherence* (how semantically tight those assignments are).

### What `tau_edge` controls

During `predict()`, each post queries its `kq` nearest neighbours in the HNSW index.
Only neighbours with cosine similarity **≥ tau_edge** are allowed to cast votes.
A higher threshold keeps only very tight neighbours → fewer but more confident assignments.

### Strategy

1. Embed a sample corpus with `multilingual-e5-large-instruct`
2. `fit()` the model once
3. Sweep `tau_edge` over a grid, collecting per-tau metrics
4. Plot the tradeoff curves and pick an elbow

---

## 0 · Install / imports

In [ ]:
# If running in a fresh env:
# !pip install narrativecluster sentence-transformers matplotlib seaborn pandas tqdm

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path

from narrativecluster import NarrativeClusterer, ClusterConfig

sns.set_theme(style='whitegrid', font_scale=1.15)
RNG = np.random.default_rng(42)

## 1 · Embed a sample corpus with multilingual-e5-large-instruct

For the tau sweep we only need a few thousand posts — a random sample from your
full dataset is fine.  We use the **instruct** variant of E5 which needs a
one-sentence task description prepended to each query.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = 'intfloat/multilingual-e5-large-instruct'

# Task description — tune this to your domain
E5_TASK = (
    'Given a social-media post, retrieve semantically similar posts '
    'that discuss the same narrative or claim'
)

def e5_encode(model: SentenceTransformer, texts: list[str], batch_size: int = 64) -> np.ndarray:
    """
    Encode texts with the multilingual-e5-large-instruct format:
      - Queries get the 'Instruct: ... \nQuery: ...' prefix
      - Documents (corpus) are passed as-is
    
    For clustering we treat every post as a *query* so they share the
    same instruction-tuned representation.
    """
    prefixed = [f'Instruct: {E5_TASK}\nQuery: {t}' for t in texts]
    vecs = model.encode(
        prefixed,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return vecs.astype(np.float32)

print(f'Loading {EMBED_MODEL} …')
embed_model = SentenceTransformer(EMBED_MODEL)
print('Done.')

In [ ]:
# ── Option A: load your real data ────────────────────────────────────────────
# df_raw = pd.read_parquet('annotation_inputs/df_claim_level.parquet')
# df_sample = df_raw.sample(n=5_000, random_state=42).reset_index(drop=True)

# ── Option B: synthetic data for demo purposes ───────────────────────────────
# We simulate 5 tight narrative clusters + noise by building sentences that
# share a strong lexical anchor within each cluster.

NARRATIVES = [
    ("election fraud",   ["ballot stuffing", "rigged voting machines", "stolen election",
                          "mail-in fraud", "voter suppression", "fake votes"]),
    ("covid vaccine",    ["mRNA side effects", "spike protein danger", "vaccine mandate",
                          "natural immunity", "myocarditis risk", "booster harm"]),
    ("immigration",      ["open borders", "illegal crossings", "deportation policy",
                          "asylum seekers", "caravan threat", "border wall"]),
    ("climate change",   ["carbon emissions", "fossil fuel lobby", "green new deal",
                          "arctic melting", "extreme weather", "net zero target"]),
    ("media bias",       ["fake news", "mainstream media lies", "censorship agenda",
                          "shadow banning", "fact-check bias", "propaganda outlet"]),
]

PLATFORMS = ['twitter', 'telegram', 'gab', '4chan', 'truthsocial']
TEMPLATES = [
    "Breaking: {anchor} is spreading across {platform}.",
    "They don't want you to know about {anchor}.",
    "The truth about {anchor} that nobody talks about.",
    "Share this before {anchor} gets deleted!",
    "PROOF: {anchor} has been confirmed by insiders.",
    "Why is {anchor} being suppressed by the mainstream?",
    "New report: {anchor} is worse than we thought.",
    "Wake up people — {anchor} is real and happening now.",
]

rows = []
for cluster_name, anchors in NARRATIVES:
    for _ in range(800):          # 800 posts per narrative
        anchor  = RNG.choice(anchors)
        tmpl    = RNG.choice(TEMPLATES)
        platform = RNG.choice(PLATFORMS, p=[0.40, 0.25, 0.15, 0.12, 0.08])
        rows.append({'text': tmpl.format(anchor=anchor, platform=platform),
                     'platform': platform,
                     'true_cluster': cluster_name})

# Add ~500 noise posts
noise_words = ['weather today', 'recipe idea', 'sports score', 'local event',
                'movie review', 'pet photo', 'travel tip', 'book club']
for _ in range(500):
    w = RNG.choice(noise_words)
    rows.append({'text': f'Just saw something about {w} today!',
                 'platform': RNG.choice(PLATFORMS),
                 'true_cluster': 'noise'})

df_sample = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Sample size: {len(df_sample):,} posts')
df_sample['true_cluster'].value_counts()

In [ ]:
# Encode with E5
vecs = e5_encode(embed_model, df_sample['text'].tolist())
df_sample['embedding'] = list(vecs)
print(f'Embedding shape: {vecs.shape}')

## 2 · Fit the clusterer (once)

We fit once and then reuse the fitted model for every tau in the sweep.
Only `predict()` changes between sweeps.

In [ ]:
BASE_CFG = ClusterConfig(
    embed_col='embedding',
    text_col='text',
    site_col='platform',
    # Graph-building / Leiden — these don't change during the tau sweep
    k_graph=32,
    tau_graph=0.60,   # slightly relaxed for the synthetic data
    cap_m=16,
    leiden_resolution=1.0,
    # Prediction params — tau_edge is what we'll sweep
    kq=40,
    tau_edge=0.65,   # placeholder; overridden in the sweep
    min_cluster_size=4,
    query_chunk=1_000,
)

CKPT = Path('/tmp/nc_tau_sweep_ckpt')
model = NarrativeClusterer(config=BASE_CFG, checkpoint_dir=CKPT)
model.fit(df_sample)

In [ ]:
# Quick sanity check on Leiden output
diag = model.cluster_diagnostics()
print(f"Total clusters: {len(diag)}")
print(f"Clusters with ≥4 posts: {diag['eligible'].sum()}")
diag.head(10)

## 3 · Sweep `tau_edge` and collect metrics

For each candidate threshold we measure:

| Metric | Definition |
|--------|------------|
| **coverage** | Fraction of posts that received a confident assignment (`nv_accepted == True`) |
| **mean_vote_frac** | Average top-cluster vote fraction among accepted posts |
| **mean_best_sim** | Average cosine similarity to the nearest same-cluster neighbour |
| **intra_sim** | Mean pairwise cosine similarity *within* each assigned cluster (**proxy for coherence**) — sampled for speed |
| **purity** (optional) | If ground-truth labels are available: fraction of accepted posts where pred matches true label |

In [ ]:
import copy
from sklearn.preprocessing import normalize as sk_normalize

def intra_cluster_sim(df_pred: pd.DataFrame,
                      embed_col: str = 'embedding',
                      max_per_cluster: int = 50,
                      rng: np.random.Generator = RNG) -> float:
    """Mean cosine similarity between all pairs within each predicted cluster.
    Sampled to keep runtime O(n) rather than O(n²)."""
    accepted = df_pred[df_pred['nv_accepted']]
    if len(accepted) == 0:
        return np.nan
    sims = []
    for cid, grp in accepted.groupby('nv_pred_cluster'):
        if len(grp) < 2:
            continue
        idx = rng.choice(len(grp), size=min(len(grp), max_per_cluster), replace=False)
        vecs = np.vstack(grp.iloc[idx][embed_col].values).astype(np.float32)
        vecs = sk_normalize(vecs)
        gram = vecs @ vecs.T
        # upper triangle only (excluding diagonal)
        upper = gram[np.triu_indices(len(vecs), k=1)]
        sims.append(float(upper.mean()))
    return float(np.mean(sims)) if sims else np.nan


def purity_score(df_pred: pd.DataFrame, true_col: str = 'true_cluster') -> float:
    """Cluster purity against a ground-truth column (if available)."""
    if true_col not in df_pred.columns:
        return np.nan
    accepted = df_pred[df_pred['nv_accepted']]
    if len(accepted) == 0:
        return np.nan
    correct = 0
    for cid, grp in accepted.groupby('nv_pred_cluster'):
        correct += grp[true_col].value_counts().iloc[0]
    return correct / len(accepted)

In [ ]:
TAU_GRID = np.arange(0.50, 0.85, 0.025).round(3).tolist()

records = []
for tau in tqdm(TAU_GRID, desc='tau_edge sweep'):
    # Patch only the prediction-time params; graph / Leiden stay fixed
    model.config.tau_edge = tau

    df_pred = model.predict(df_sample)

    accepted      = df_pred['nv_accepted']
    coverage      = float(accepted.mean())
    mean_vfrac    = float(df_pred.loc[accepted, 'nv_vote_frac'].mean()) if accepted.any() else np.nan
    mean_bsim     = float(df_pred.loc[accepted, 'nv_best_sim'].mean()) if accepted.any() else np.nan
    coherence     = intra_cluster_sim(df_pred)
    purity        = purity_score(df_pred)
    n_assigned    = int(accepted.sum())
    n_pred_clust  = int(df_pred.loc[accepted, 'nv_pred_cluster'].nunique()) if accepted.any() else 0

    records.append(dict(
        tau_edge=tau,
        coverage=coverage,
        n_assigned=n_assigned,
        n_pred_clusters=n_pred_clust,
        mean_vote_frac=mean_vfrac,
        mean_best_sim=mean_bsim,
        intra_sim=coherence,
        purity=purity,
    ))

sweep = pd.DataFrame(records)
sweep

## 4 · Visualise the tradeoff curves

Look for the **elbow** where coverage starts falling steeply but coherence has already plateaued — that's your sweet spot.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
fig.suptitle('tau_edge sweep — coverage / coherence tradeoff', fontsize=14, fontweight='bold')

METRICS = [
    ('coverage',       'Coverage (fraction assigned)', '#2196F3'),
    ('intra_sim',      'Intra-cluster cosine sim (coherence)', '#4CAF50'),
    ('mean_vote_frac', 'Mean vote fraction', '#FF9800'),
    ('mean_best_sim',  'Mean best-neighbour sim', '#9C27B0'),
    ('purity',         'Cluster purity (vs true labels)', '#E91E63'),
    ('n_pred_clusters','# unique predicted clusters', '#607D8B'),
]

for ax, (col, label, color) in zip(axes.flat, METRICS):
    ax.plot(sweep['tau_edge'], sweep[col], marker='o', color=color, linewidth=2)
    ax.set_xlabel('tau_edge')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    # Mark the coverage elbow (largest drop between consecutive taus)
    if col == 'coverage':
        drops = np.diff(sweep['coverage'].values)
        elbow_idx = int(np.argmin(drops)) + 1
        elbow_tau = sweep['tau_edge'].iloc[elbow_idx]
        ax.axvline(elbow_tau, color='red', linestyle='--', alpha=0.7,
                   label=f'Elbow @ {elbow_tau:.3f}')
        ax.legend(fontsize=9)

plt.savefig('/tmp/tau_sweep.png', dpi=150)
plt.show()
print(f'\nCoverage elbow at tau_edge = {elbow_tau:.3f}')

## 5 · Composite score: coverage × coherence

A simple F1-style geometric mean of the two primary objectives.

In [ ]:
# Min-max normalise both metrics to [0, 1] before combining
def minmax(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo + 1e-12)

sweep['cov_norm']  = minmax(sweep['coverage'])
sweep['coh_norm']  = minmax(sweep['intra_sim'])

# Geometric mean  (= harmonic mean of the logs → penalises imbalance)
sweep['composite'] = np.sqrt(sweep['cov_norm'] * sweep['coh_norm'])

best_row = sweep.loc[sweep['composite'].idxmax()]
best_tau = float(best_row['tau_edge'])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sweep['tau_edge'], sweep['composite'], 'ko-', linewidth=2, label='composite')
ax.plot(sweep['tau_edge'], sweep['cov_norm'],  '--', color='#2196F3', alpha=0.6, label='coverage (norm)')
ax.plot(sweep['tau_edge'], sweep['coh_norm'],  '--', color='#4CAF50', alpha=0.6, label='coherence (norm)')
ax.axvline(best_tau, color='red', linestyle=':', linewidth=1.8, label=f'Best: {best_tau:.3f}')
ax.set_xlabel('tau_edge');  ax.set_ylabel('Score (normalised)')
ax.set_title('Composite score = √(coverage × coherence)')
ax.legend();  plt.tight_layout()
plt.savefig('/tmp/tau_composite.png', dpi=150)
plt.show()

print(f'\n✅  Recommended tau_edge = {best_tau:.3f}')
print(best_row.to_string())

## 6 · Per-site tau sensitivity

Some platforms (e.g. Telegram channels) have naturally higher intra-community
cosine similarity because users speak in a narrow shared vocabulary.  Check
whether the global optimum is stable across platforms.

In [ ]:
site_records = []

for tau in tqdm(TAU_GRID[::2], desc='per-site sweep (every other tau)'):
    model.config.tau_edge = tau
    df_pred = model.predict(df_sample)
    _, tbl = model.site_stats(df_pred)
    tbl = tbl.reset_index()
    tbl['tau_edge'] = tau
    site_records.append(tbl)

site_sweep = pd.concat(site_records, ignore_index=True)

fig, ax = plt.subplots(figsize=(11, 5))
palette = sns.color_palette('tab10', n_colors=site_sweep['site'].nunique())
for color, (site, grp) in zip(palette, site_sweep.groupby('site')):
    ax.plot(grp['tau_edge'], grp['accept_rate'], marker='o', label=site,
            color=color, linewidth=1.8)

ax.axvline(best_tau, color='red', linestyle=':', label=f'Best global: {best_tau:.3f}')
ax.set_xlabel('tau_edge');  ax.set_ylabel('Acceptance rate')
ax.set_title('Per-platform acceptance rate vs tau_edge')
ax.legend(ncol=2, fontsize=9);  plt.tight_layout()
plt.savefig('/tmp/tau_per_site.png', dpi=150)
plt.show()

## 7 · Inspect assignments at the chosen tau

A quick look at which posts were accepted vs rejected.

In [ ]:
model.config.tau_edge = best_tau
df_final = model.predict(df_sample)

print(f'tau_edge = {best_tau}')
print(f'Accepted: {df_final["nv_accepted"].sum():,} / {len(df_final):,} '
      f'({df_final["nv_accepted"].mean():.1%})')

# Vote-fraction histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_acc = df_final[df_final['nv_accepted']]
axes[0].hist(df_acc['nv_vote_frac'], bins=30, color='#2196F3', edgecolor='white')
axes[0].set_title('Vote fraction (accepted posts)')
axes[0].set_xlabel('nv_vote_frac');  axes[0].set_ylabel('count')

axes[1].hist(df_acc['nv_best_sim'], bins=30, color='#4CAF50', edgecolor='white')
axes[1].set_title('Best-neighbour cosine sim (accepted posts)')
axes[1].set_xlabel('nv_best_sim');  axes[1].set_ylabel('count')

plt.tight_layout()
plt.savefig('/tmp/tau_final_hist.png', dpi=150)
plt.show()

In [ ]:
# Sample accepted posts per cluster
(
    df_final[df_final['nv_accepted']]
    .groupby('nv_pred_cluster')
    .apply(lambda g: g.sample(min(3, len(g)), random_state=42))[['text', 'platform', 'nv_vote_frac', 'nv_best_sim']]
    .head(30)
)

## 8 · Summary and recommendation

Copy the recommended value into your `ClusterConfig`:

In [ ]:
print('=== Tau-edge tuning summary ===')
print(f'  Composite-optimal  tau_edge = {best_tau:.3f}')
print(f'  Coverage-elbow     tau_edge = {elbow_tau:.3f}')
print()
print('Paste into your ClusterConfig:')
print(f'''
cfg = ClusterConfig(
    embed_col='embedding',
    text_col='cleaned_text',
    site_col='site',
    tau_graph=0.68,        # graph-building threshold (separate from tau_edge)
    tau_edge={best_tau:.3f},        # <-- neighbour-vote threshold
    kq=50,
    min_cluster_size=4,
)
''')
print('\n💡 Rule of thumb for multilingual-e5-large-instruct:')
print('   E5 cosine sims tend to be higher than older models, so tau_edge')
print('   in the 0.68-0.78 range is typical.  If you see very low coverage')
print('   (<30%), lower tau_graph first (graph build), not just tau_edge.')